# Telco churn — exploratory data analysis

Dataset: [IBM Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
Place the raw CSV at `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv` before running.

Sections
1. Load & inspect
2. Class balance
3. Numeric feature distributions
4. Categorical feature breakdown
5. Correlation heatmap

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import load_data, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COLUMN

sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

## 1. Load & inspect

In [ ]:
X, y = load_data(DATA_PATH)
df = X.copy()
df[TARGET_COLUMN] = y

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Class balance

In [ ]:
counts = y.value_counts()
print(counts)
print(f'Churn rate: {y.mean():.1%}')

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(['Retained', 'Churned'], counts.values, color=['steelblue', 'tomato'])
ax.set_ylabel('Count')
ax.set_title('Class balance')
plt.tight_layout()
plt.show()

## 3. Numeric feature distributions

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(12, 3))
for ax, col in zip(axes, NUMERIC_FEATURES):
    for label, grp in df.groupby(TARGET_COLUMN):
        grp[col].dropna().hist(ax=ax, alpha=0.6, bins=30,
                               label='Churned' if label else 'Retained')
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle('Numeric features by churn status', y=1.02)
plt.tight_layout()
plt.show()

## 4. Categorical feature breakdown

In [ ]:
n_cols = 4
n_rows = -(-len(CATEGORICAL_FEATURES) // n_cols)  # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for ax, col in zip(axes, CATEGORICAL_FEATURES):
    churn_rate = df.groupby(col)[TARGET_COLUMN].mean().sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=ax, color='tomato', alpha=0.8)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Churn rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=7)
    ax.axhline(y.mean(), linestyle='--', color='gray', linewidth=0.8, label='avg')
    ax.legend(fontsize=7)

for ax in axes[len(CATEGORICAL_FEATURES):]:
    ax.set_visible(False)

plt.suptitle('Churn rate by category', y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation heatmap (numeric)

In [ ]:
corr_cols = NUMERIC_FEATURES + [TARGET_COLUMN]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Pearson correlation')
plt.tight_layout()
plt.show()